In [ ]:
import pandas as pd
import requests
import logging
from bs4 import BeautifulSoup as bs
from pathlib import Path

data_dir = Path.cwd().parent /'data'


In [ ]:
# read the working url
df = pd.read_excel(data_dir / "updated_gaupalika_contacts.xlsx")

In [ ]:
df.shape

In [ ]:
# Remove if duplicated URL
# df = df.drop_duplicates(subset=['working_url'], keep='first')

In [ ]:
df.shape

In [ ]:
urls = df['working_url'].to_list()

In [ ]:
dfs_dict = {}


In [ ]:
len(dfs_dict.keys())

In [ ]:
# Check if response code is 200 
def check_url_response(url):

    '''The function checks if the URL is valid and returns True if the response code is 200, otherwise returns False.
    Args:
        url (str): The URL to check.
    Returns:
        bool: True if the response code is 200, False otherwise.
    '''
    try:
        response = requests.get(url)
        if response.status_code == 200:
            return True
        else:
            logging.error(f"Error: Received status code {response.status_code} for URL: {url}")
            return False
    except requests.exceptions.RequestException as e:
        logging.error(f"Request failed: {e}")
        return False

# find the div that has tables
def find_table(response):
    soup = bs(response.text, 'html.parser')
    table_divs = []
    
    divs = soup.find_all('div', class_='view-content')
    # If no div found, log an error
    if not divs:
        logging.error(f"This div does not have a view-content: {response.url}")
        return []
    if divs:
        for div in divs:
        # Check if this div has a table
            if div.find('table'):
                table_divs.append(div)
            else:
                logging.error(f"No table found in this div for URL: {response.url}")
    
    return table_divs

# if the div has a table, then read it with pd.read_html. So that it gets converted to a dataframe
def read_table(table_div):
    try:
        df = pd.read_html(table_div)
        return df
    except ValueError as e:
        logging.error(f"Error reading table: {e}")
        return None
    
# The return dataframe from read_table needs to go into a dictionary of dataframes
def store_dataframe(df, url,dfs_dict):
    '''dfs_dict is a dictionary of dataframes
    url is the key for the dictionary
    df is the dataframe to be stored
    '''
    if df is not None:
        # Use the URL as a key to store the dataframe
        dfs_dict[url] = df
    else:
        logging.error(f"Dataframe is None for URL: {url}")


    

In [ ]:
# Configure logging to include date and time in the log entries
logging.basicConfig(
    filename='error_gau.log', 
    encoding='utf-8', 
    level=logging.DEBUG,
    format='%(asctime)s - %(levelname)s - %(message)s'  # Add the date/time
)
logger = logging.getLogger(__name__)


# # test individually
# dfs = {}
# url = 'https://mangalsenmun.gov.np/ne/staff'
# if check_url_response(url) == True:
#     response = requests.get(url)
#     x = find_table(response)
#     print(len(x))
#     # the found table needs to be converted to a dataframe
#     df = pd.read_html(str(x))[0]
#     store_dataframe(df,url,dfs)


In [ ]:
urls_100 = urls[100:]

In [ ]:
urls_100[-1]

In [ ]:
# Set up logging to log errors to a file
for url in urls_100:
    try:
        if check_url_response(url) == True:
            response = requests.get(url)
        # use the find_table function to find the div that has the table
            table_divs = find_table(response)
            # Read the table into a DataFrame
            df = read_table(str(table_divs))
            # Store the DataFrame in the dictionary
            store_dataframe(df, url, dfs_dict)
        print(f"Processed URL: {url}")
    except requests.RequestException as e:
        logging.error(f"Unexecpted error with {url} : {e}")
        continue



In [ ]:
# Check the number of dataframes stored in the dictionary
print(len(dfs_dict.keys()))
# Check the last dataframe stored in the dictionary


In [ ]:

all_dfs = []

url_keys = list(dfs_dict.keys())# list to dataframe

for i in range(len(url_keys)):
    data = dfs_dict[url_keys[i]]

    if data is not None:
        try:
            df = pd.DataFrame(data[0])
            df['source_url'] = url_keys[i]  # Optional: keep track of source
            all_dfs.append(df)
        except Exception as e:
            logging.error(f"Error for {url_keys[i]}: {e}")
    else:
        logging.error(f"No data for {url_keys[i]}")

# Final combined DataFrame
final_df = pd.concat(all_dfs, ignore_index=True)

In [ ]:
final_df.head()

In [ ]:
final_df.to_excel('final_combined_data_gaupalika.xlsx', index=False)

In [ ]:
# manually corrected the headings in the exported excel file for concat, now I read the file again
df = pd.read_excel('final_combined_data_gaupalika.xlsx')

In [ ]:
df.columns

In [ ]:
# Assuming 'df' is your DataFrame
# Step 1: Identify all the columns related to 'Name', 'Designation', 'Email', 'Phone', etc.
name_columns = [col for col in df.columns if 'Name' in col]
designation_columns = [col for col in df.columns if 'Designation' in col]
email_columns = [col for col in df.columns if 'Email' in col]
phone_columns = [col for col in df.columns if 'Phone' in col]
section_columns = [col for col in df.columns if 'Section' in col]
address_columns = [col for col in df.columns if 'Address' in col]



# Step 2: Combine the columns into single columns
df['Names_combined'] = df[name_columns].apply(lambda x: ', '.join(x.dropna().astype(str)), axis=1)
df['Designations_combined'] = df[designation_columns].apply(lambda x: ', '.join(x.dropna().astype(str)), axis=1)
df['Emails_combined'] = df[email_columns].apply(lambda x: ', '.join(x.dropna().astype(str)), axis=1)
df['Phones_combined'] = df[phone_columns].apply(lambda x: ', '.join(x.dropna().astype(str)), axis=1)
df['Section_combined'] = df[section_columns].apply(lambda x: ', '.join(x.dropna().astype(str)), axis=1)
df['Address_combined'] = df[address_columns].apply(lambda x: ', '.join(x.dropna().astype(str)), axis=1)


# Step 3: Drop the original columns if no longer needed
df.drop(columns=name_columns + designation_columns + email_columns + phone_columns + section_columns + address_columns, inplace=True)




In [ ]:
df.columns = ['source_url','Names','Desgnations','Emails','Phones','Section','Address']
# Step 4: Save the final DataFrame to an Excel file
df.to_excel('final_combined_data_gaupalika_cleaned.xlsx', index=False)

In [ ]:
# Set up logging to log errors to a file
logger = logging.getLogger(__name__)

logging.basicConfig(filename='error.log', encoding='utf-8', level=logging.DEBUG)

for url in url:
    try:
        response = requests.get(url)
        if response.status_code == 200:
            if 'view-content' in response.text:
                try:
                    
                    table_found = pd.read_html(url)
                    if table_found:
                        print(table_found)
                        df = table_found[0]
                        df["url"] = url
                        dfs_dict[url] = df
                    else:
                        logging.debug(f"Not found 'view-content' at {url}")
                except ValueError:
                    logging.debug(f"Error reading table from {url}")
                    logging.debug(f"Error reading table from {url}")

    except requests.RequestException as e:
        logging.error(f"Unexecpted error with {url} : {e}")
        continue
